In [64]:
import torch
import torch.nn.functional as F
from torchvision import transforms
import torch.nn as nn

import matplotlib.pyplot as plt
import math

import numpy as np
from PIL import Image
from pathlib import Path
from scipy.ndimage import convolve
from scipy.fft import dctn

In [65]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [66]:
CROP_SIZE   = 256
N_CLASSES   = 5
BATCH_SIZE  = 64           
EPOCHS      = 20
LR_INIT     = 3e-3
WEIGHT_DECAY = 1e-4
DROPOUT     = 0.45

In [67]:
KERNELS_SRM = [
    np.array([[ 0,  1,  0], [ 1, -4,  1], [ 0,  1,  0]], dtype=np.float32) / 4.0,
    np.array([[ 1,  1,  1], [ 1, -8,  1], [ 1,  1,  1]], dtype=np.float32) / 8.0,
    np.array([[ 0,  0,  0], [ 1, -2,  1], [ 0,  0,  0]], dtype=np.float32) / 2.0,
    np.array([[ 0,  1,  0], [ 0, -2,  0], [ 0,  1,  0]], dtype=np.float32) / 2.0,
    np.array([[ 1,  0,  1], [ 0, -4,  0], [ 1,  0,  1]], dtype=np.float32) / 4.0,
]

def calcular_residuos_srm(img_array, T=4.0):
    img = img_array.astype(np.float32)
    canales = []
    for c in range(3):
        for k in KERNELS_SRM:
            r = convolve(img[:, :, c], k, mode='reflect')
            canales.append(np.clip(r, -T, T) / T)
    return np.stack(canales, axis=-1)  # (H, W, 15)

def dct_features(img_array, block_size=8):
    H, W = img_array.shape[:2]
    img_f = img_array.astype(np.float32)
    Y = 0.299*img_f[:,:,0] + 0.587*img_f[:,:,1] + 0.114*img_f[:,:,2] - 128.0
    dct_map = np.zeros((H, W), dtype=np.float32)
    for i in range(0, H - block_size + 1, block_size):
        for j in range(0, W - block_size + 1, block_size):
            bloque = Y[i:i+block_size, j:j+block_size]
            D = dctn(bloque, norm='ortho')
            dct_map[i:i+block_size, j:j+block_size] = np.log1p(np.abs(D))
    mx = dct_map.max()
    if mx > 0:
        dct_map /= mx
    return dct_map[:, :, np.newaxis]

In [68]:
# ── Arquitectura del Modelo ──────────────────────────────────────────────────
class ConvBNMish(nn.Module):
    def __init__(self, in_c, out_c, kernel=3, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel, stride=stride, padding=kernel//2, bias=False),
            nn.BatchNorm2d(out_c),
            nn.Mish()
        )
    def forward(self, x): return self.block(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.downsample = (stride != 1 or in_c != out_c)
        if self.downsample:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
        self.conv1 = ConvBNMish(in_c, out_c, 3, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c)
        )
        self.act = nn.Mish()
    def forward(self, x):
        shortcut = self.shortcut(x) if self.downsample else x
        out = self.conv1(x)
        out = self.conv2(out)
        return self.act(out + shortcut)

class SeparableConv2d(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.depthwise = nn.Conv2d(in_c, in_c, 3, padding=1, groups=in_c, bias=False)
        self.pointwise = nn.Conv2d(in_c, out_c, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.Mish()
    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))

class SqueezeExcitation(nn.Module):
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, max(channels // ratio, 4)),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels // ratio, 4), channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        flat = x.view(b, c, -1).mean(dim=2)
        scale = self.fc(flat).view(b, c, 1, 1)
        return x * scale

class StegoCNNv5(nn.Module):
    def __init__(self):
        super().__init__()
        self.srm_stem = ConvBNMish(15, 32)
        self.srm_layers = nn.Sequential(
            ResidualBlock(32, 32),   nn.AvgPool2d(2),
            ResidualBlock(32, 64),   nn.AvgPool2d(2),
            ResidualBlock(64, 96),   nn.AvgPool2d(2),
            ResidualBlock(96, 128),  nn.AvgPool2d(2),
            ResidualBlock(128, 128)
        )
        
        self.dct_stem = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.Mish())
        self.dct_layers = nn.Sequential(
            SeparableConv2d(16, 32),  nn.AvgPool2d(2),
            SeparableConv2d(32, 64),  nn.AvgPool2d(2),
            SeparableConv2d(64, 96),  nn.AvgPool2d(2),
            SeparableConv2d(96, 128), nn.AvgPool2d(2),
            SeparableConv2d(128, 128)
        )
        
        self.se = SqueezeExcitation(256, ratio=8)
        
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.Mish(), nn.Dropout(DROPOUT),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.Mish(), nn.Dropout(DROPOUT),
            nn.Linear(64, N_CLASSES)
        )

    def forward(self, srm, dct):
        f_srm = self.srm_layers(self.srm_stem(srm)).mean(dim=[2, 3])
        f_dct = self.dct_layers(self.dct_stem(dct)).mean(dim=[2, 3])
        fused = torch.cat([f_srm, f_dct], dim=1)
        
        fused_tensor = fused.view(fused.size(0), fused.size(1), 1, 1)
        fused_scaled = self.se(fused_tensor).view(fused.size(0), -1)
        
        return self.head(fused_scaled)

In [69]:
NOMBRES_CLASES = ['Cover', 'LSB', 'DCT', 'PVD', 'BPCS']

In [70]:
modelo = StegoCNNv5()
state_dict = torch.load("stego_cnn_v5_best.pt", map_location=torch.device(device))
modelo.load_state_dict(state_dict)
modelo.to(device)

StegoCNNv5(
  (srm_stem): ConvBNMish(
    (block): Sequential(
      (0): Conv2d(15, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): Mish()
    )
  )
  (srm_layers): Sequential(
    (0): ResidualBlock(
      (conv1): ConvBNMish(
        (block): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): Mish()
        )
      )
      (conv2): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (act): Mish()
    )
    (1): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (2): ResidualBlock(
      (shortcut): Sequential(
      

In [71]:
modelo.eval()

StegoCNNv5(
  (srm_stem): ConvBNMish(
    (block): Sequential(
      (0): Conv2d(15, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): Mish()
    )
  )
  (srm_layers): Sequential(
    (0): ResidualBlock(
      (conv1): ConvBNMish(
        (block): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): Mish()
        )
      )
      (conv2): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (act): Mish()
    )
    (1): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (2): ResidualBlock(
      (shortcut): Sequential(
      

In [72]:
# Transform para inferencia: sin flips ni rotaciones aleatorias
inference_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # igual que en 'transform'
])

In [73]:
images = []

images.append("images/deidad.png")
images.append("images/elfuturo.png")
images.append("images/pridemonth.png")
images.append("images/vege.png")
images.append("images/wigetta.png")

images.append("images/results/original/img_01_og.png")
images.append("images/results/original/img_02_og.png")
images.append("images/results/original/img_03_og.png")
images.append("images/results/original/img_04_og.png")
images.append("images/results/original/img_05_og.png")

images.append("images/results/lsb/img_01_lsb.png")
images.append("images/results/lsb/img_02_lsb.png")
images.append("images/results/lsb/img_03_lsb.png")
images.append("images/results/lsb/img_04_lsb.png")
images.append("images/results/lsb/img_05_lsb.png")

images.append("images/results/dct/img_01_dct.png")
images.append("images/results/dct/img_02_dct.png")
images.append("images/results/dct/img_03_dct.png")
images.append("images/results/dct/img_04_dct.png")
images.append("images/results/dct/img_05_dct.png")

images.append("images/results/pvd/img_01_pvd.png")
images.append("images/results/pvd/img_02_pvd.png")
images.append("images/results/pvd/img_03_pvd.png")
images.append("images/results/pvd/img_04_pvd.png")
images.append("images/results/pvd/img_05_pvd.png")

images.append("images/results/dct/img_01_dct.png")
images.append("images/results/dct/img_02_dct.png")
images.append("images/results/dct/img_03_dct.png")
images.append("images/results/dct/img_04_dct.png")
images.append("images/results/dct/img_05_dct.png")

In [74]:
def predecir_imagen(ruta_imagen: str):
    img = Image.open(ruta_imagen).convert('RGB')
    arr = np.array(img, dtype=np.uint8)
    H, W, _ = arr.shape

    # Crop central de 256x256 (igual que en el Dataset de val/test)
    y0 = (H - CROP_SIZE) // 2
    x0 = (W - CROP_SIZE) // 2
    parche = arr[y0:y0+CROP_SIZE, x0:x0+CROP_SIZE, :]

    # Preprocesamiento
    srm = np.transpose(calcular_residuos_srm(parche), (2, 0, 1)).astype(np.float32)
    dct = np.transpose(dct_features(parche), (2, 0, 1)).astype(np.float32)

    srm_t = torch.tensor(srm).unsqueeze(0).to(device)  # (1, 15, 256, 256)
    dct_t = torch.tensor(dct).unsqueeze(0).to(device)  # (1, 1,  256, 256)

    with torch.no_grad():
        logits = modelo(srm_t, dct_t)
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
        pred   = np.argmax(probs)

    print(f"Predicción: {NOMBRES_CLASES[pred]}")
    print("Probabilidades por clase:")
    for nombre, p in zip(NOMBRES_CLASES, probs):
        print(f"  {nombre:8s}: {p*100:.2f}%")
    return pred, probs

In [75]:
for i in range(len(images)):
    print(f"\n=== Prediciendo imagen {i+1}/{len(images)}: {images[i]} ===")
    predecir_imagen(images[i])


=== Prediciendo imagen 1/30: images/deidad.png ===
Predicción: Cover
Probabilidades por clase:
  Cover   : 90.05%
  LSB     : 0.26%
  DCT     : 0.06%
  PVD     : 7.38%
  BPCS    : 2.25%

=== Prediciendo imagen 2/30: images/elfuturo.png ===
Predicción: Cover
Probabilidades por clase:
  Cover   : 91.29%
  LSB     : 0.43%
  DCT     : 0.09%
  PVD     : 5.73%
  BPCS    : 2.47%

=== Prediciendo imagen 3/30: images/pridemonth.png ===
Predicción: Cover
Probabilidades por clase:
  Cover   : 100.00%
  LSB     : 0.00%
  DCT     : 0.00%
  PVD     : 0.00%
  BPCS    : 0.00%

=== Prediciendo imagen 4/30: images/vege.png ===
Predicción: Cover
Probabilidades por clase:
  Cover   : 98.31%
  LSB     : 0.39%
  DCT     : 0.04%
  PVD     : 0.51%
  BPCS    : 0.75%

=== Prediciendo imagen 5/30: images/wigetta.png ===
Predicción: Cover
Probabilidades por clase:
  Cover   : 100.00%
  LSB     : 0.00%
  DCT     : 0.00%
  PVD     : 0.00%
  BPCS    : 0.00%

=== Prediciendo imagen 6/30: images/results/original/img_